# avgpool-reduce — worked example 1: 1D average pooling over non-overlapping windows via einops.reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `avgpool-reduce`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Average pooling reduces each window to its mean. The einops trick is to **factor** the axis you want to pool into `(out_axis inner)`, then drop the inner factor with `'mean'`. For 1D signals shaped `(B, C, L)` with pool size `p` (and `L % p == 0`), factoring `L -> (l p1)` and reducing `p1` gives a `(B, C, L // p)` output.

## Worked solution

**Step 1 — name the shape contract.** Input is `(B, C, L)`, pool size `p`, output is `(B, C, L // p)`. Each output entry is the mean of a contiguous block of `p` samples.

**Step 2 — pick the factoring.** We want to split the length axis `L` into `l` blocks of `p` each. In einops that is the source pattern `b c (l p1)`, where `l = L // p` and `p1 = p`. The key idea: `(l p1)` means 'this axis is `l * p1` and I am telling you `p1`, so infer `l`'.

**Step 3 — drop the inner factor.** On the right side we write `b c l`. Because `p1` is ABSENT from the right side, einops reduces over it. Passing `'mean'` makes that reduction an average — exactly window-mean pooling.

**Step 4 — tell einops the factor size.** We must pass `p1=p` so the library can solve `l = L // p`. Without it the factoring is ambiguous.

**Step 5 — verify against torch.** `F.avg_pool1d(x, kernel_size=p)` is the ground truth for non-overlapping 1D average pooling, so we compare to fp tolerance.

In [ ]:
import torch as t
import torch.nn.functional as F
import einops
from torch import Tensor

t.manual_seed(0)

def avgpool1d_via_reduce(x: Tensor, p: int) -> Tensor:
    return einops.reduce(x, 'b c (l p1) -> b c l', 'mean', p1=p)

x = t.randn(2, 3, 12)
out = avgpool1d_via_reduce(x, 4)
ref = F.avg_pool1d(x, kernel_size=4)
print('out shape:', tuple(out.shape))
print('matches F.avg_pool1d:', bool(t.allclose(out, ref, atol=1e-6)))
print('first window mean:', out[0, 0, 0].item())